# 04 spacyNER vs GLiNER — Comparación de entidades

🎯 **Objetivo:** Ver qué entidades detecta cada modelo (GLiNER y spaCy), comparar resultados y así analizar qué aporta cada modelo al pipeline.

🔍 **Puntos clave:**
1. Carga de resultados previos spaCy y GliNER
2. Comparación resultados spaCy vs GLiNER

## 0 · Imports y configuración

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd

# Agrega la raíz del proyecto al path para importar src/
PROJECT_ROOT = Path("../").resolve()  # ajusta si se corre desde otro lugar
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
SPACY_CANDIDATES   = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_spacy"
GLINER_CANDIDATES  = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_gliner"

# ── Parámetros ─────────────────────────────────────────────────────────────
ENCODING        = "utf-8"

print(f"Results spaCy:   {SPACY_CANDIDATES}")
print(f"Results GliNER:  {GLINER_CANDIDATES}")

## 1 · Resultados spaCy y GliNER previos

In [ ]:
# Carga de candidatos spaCy (del notebook 02) / GliNER (del notebook 03) para comparar después
# Si no corriste el notebook 02/03 aún, esta celda mostrará un aviso y continuará
def cargar_resultados_previos(tipo: str) -> dict[str, dict]:

    candidates_dir = SPACY_CANDIDATES if tipo == "spaCy" else GLINER_CANDIDATES

    entities_data: dict[str, dict] = {}
    
    for json_path in sorted(candidates_dir.glob("*_candidates.json")):
        with open(json_path, encoding=ENCODING) as f:
            data = json.load(f)
        entities_data[data["doc_id"]] = data
    
    if entities_data:
        print(f"Candidatos {tipo} cargados: {len(entities_data)} documentos")
    else:
        number_notebook = "02" if tipo == "spaCy" else "03"
        print(f"⚠️  No se encontraron JSONs de {tipo}. "
              f"Corre el notebook {number_notebook} primero para la comparación, "
              "no puedes seguir sin este paso.")
    
    return entities_data

In [ ]:
spacy_data = cargar_resultados_previos("spaCy")

In [ ]:
gliner_data = cargar_resultados_previos("GliNER")

In [ ]:
spacy_data['C1_transcript']['spacy_candidates'][0].keys()

## 2 · Comparación resultados spaCy vs GLiNER

In [ ]:
# Reconstruir dataframe desde los JSONs cargados (entidades exportadas)
def build_entity_df(source_data, source_name: str) -> pd.DataFrame:
    if source_data:
        rows = []
        for doc_id, data in source_data.items():
            for e in data[f"{source_name}_candidates"]:
                rows.append({
                    "doc_id":     doc_id,
                    "texto":      e["text"],
                    "texto_norm": e["text"].lower().strip(),
                    "tipo":       e["label"],
                    "start":      e["start"],
                    "end":        e["end"],
                    "score":      e["score"],
                    "fuente":     source_name,
                })
        df_results = pd.DataFrame(rows)
    else:
        df_results = pd.DataFrame()  # vacío si no hay datos spaCy o GliNER
    return df_results

In [ ]:
df_spacy = build_entity_df(spacy_data, "spacy")
df_gliner = build_entity_df(gliner_data, "gliner")

print(f"spaCy  → {len(df_spacy)} entidades")
print(f"GLiNER → {len(df_gliner)} entidades")

In [ ]:
# Tabla comparativa de entidades por tipo
comp = pd.DataFrame({
    "spaCy":  df_spacy["tipo"].value_counts(),
    "GLiNER": df_gliner["tipo"].value_counts(),
}).fillna(0).astype(int)
comp["diferencia"] = comp["GLiNER"] - comp["spaCy"]
display(comp)

In [ ]:
# Solapamiento: ¿cuántas entidades (por texto normalizado) detectan ambos?
def revisar_entidades(df_spacy: pd.DataFrame, df_gliner: pd.DataFrame, documento_revisar=None):

    if documento_revisar:
        print(f"Documento             : {documento_revisar:>4}")
        spacy_set  = set(df_spacy[df_spacy["doc_id"] == documento_revisar]["texto_norm"].unique())
        gliner_set = set(df_gliner[df_gliner["doc_id"] == documento_revisar]["texto_norm"].unique())
    else:
        print(f"Documento             : Todos")
        spacy_set  = set(df_spacy["texto_norm"].unique())
        gliner_set = set(df_gliner["texto_norm"].unique())

    solo_spacy  = spacy_set  - gliner_set
    solo_gliner = gliner_set - spacy_set
    ambos       = spacy_set  & gliner_set

    print(f"Detectadas por ambos  : {len(ambos):>4}")
    print(f"Solo spaCy            : {len(solo_spacy):>4}")
    print(f"Solo GLiNER           : {len(solo_gliner):>4}")

    print(f"\n── Ambos (primeras 20) ──")
    print(sorted(ambos)[:20])

    print(f"\n── Solo en spaCy (primeras 20) ──")
    print(sorted(solo_spacy)[:20])

    print(f"\n── Solo en GLiNER (primeras 20) ──")
    print(sorted(solo_gliner)[:20])

In [ ]:
revisar_entidades(df_spacy, df_gliner)

In [ ]:
revisar_entidades(df_spacy, df_gliner, documento_revisar="C4_transcript")